# Importando as Bibliotecas

In [ ]:
# instalei as principais bibliotecas para análise e formatação dos dados
%pip install pandas numpy matplotlib seaborn

In [ ]:
# importei a biblioteca pandas
import pandas as pd

# importei a biblioteca unicodedata para normalização de texto
import unicodedata

# criei a conexão do banco de dados na memória com o sql 
import sqlite3

conn = sqlite3.connect(":memory:")

# importei a biblioteca time para controle de tempo
import time

# importei a biblioteca requests para requisições HTTP
import requests

# importei a biblioteca numpy para operações numéricas
import numpy as np


# Tabela clientes

In [ ]:
# carreguei os dados do dataset clientes e visualizei as primeiras linhas
df_clientes = pd.read_json('../dados_brutos/clientes_crm.json')

df_clientes.head()

In [ ]:
# explorei a estrutura e as estatísticas do dataset de clientes
df_clientes.shape
df_clientes.info()
df_clientes.describe()

In [ ]:
# verifiquei valores nulos no dataset de clientes
df_clientes.isnull().sum()

In [ ]:
# renomeei as colunas para padronzar os dados
df_clientes = df_clientes.rename(columns={
    'full_name': 'nome_completo',
    'location': 'localizacao',
    'code': 'id_cliente'
})

# verifiquei a nova condição
df_clientes.head()

In [ ]:
# organizei a ordem das colunas do dataset clientes
df_clientes = df_clientes[['id_cliente', 'nome_completo', 'localizacao', 'email']]

#Verificando a nova condição
df_clientes.head()

In [ ]:
# verifiquei a unicidade da chave id_cliente
df_clientes['id_cliente'].is_unique

In [ ]:
# padronizei os nomes removendo os espaços e ajustando a capitalização
df_clientes['nome_completo'] = df_clientes['nome_completo'].str.strip()
df_clientes['nome_completo'] = df_clientes['nome_completo'].str.title()

In [ ]:
# removi registros duplicados do dataset de clientes
df_clientes = df_clientes.drop_duplicates()

df_clientes.shape

In [ ]:
# verifiquei as colunas do dataset de clientes
df_clientes.columns

In [ ]:
# analisei os valores presentes na coluna localizacao para padronizar
df_clientes['localizacao'].unique()

In [ ]:
# defini um dicionário de estados
ufs = ['AC','AL','AP','AM','BA','CE','DF','ES','GO','MA','MT','MS','MG',
       'PA','PB','PR','PE','PI','RJ','RN','RS','RO','RR','SC','SP','SE','TO']

def limpar_loc(loc):
    if pd.isna(loc) or not loc.strip():
        return pd.Series([None, None])
    
    # normalizei e removi acentos
    loc = unicodedata.normalize('NFKD', str(loc).lower()).encode('ascii','ignore').decode('utf-8')
    
    # removi parênteses e padronizei separadores
    loc = loc.replace("(", "").replace(")", "")
    for sep in ['-', '/', '\\', ';']: loc = loc.replace(sep, ",")
    loc = loc.replace(",,",",").replace(", ",",").replace(" ,",",").strip()
    
    partes = [p.strip() for p in loc.split(",") if p.strip()]
    
    estado = None
    cidade = None
    
    # procurei UF em qualquer posição
    for i, p in enumerate(partes):
        if p.upper() in ufs:
            estado = p.upper()
            # defini que cidade é tudo que não é o estado
            cidade = " ".join([x.title() for j,x in enumerate(partes) if j != i])
            break
    
    # criei a condição de que caso não encontre UF, tentar assumir a última parte com 2 letras
    if estado is None and len(partes)>=2 and len(partes[-1])==2:
        estado = partes[-1].upper()
        cidade = " ".join(partes[:-1]).title()
    elif estado is None:
        cidade = " ".join(partes).title()
    
    return pd.Series([cidade, estado])

# apliquei no DataFrame
df_clientes[['cidade','estado']] = df_clientes['localizacao'].apply(limpar_loc)

print(df_clientes[['cidade','estado']].head(50))

In [ ]:
# reorganizei as novas colunas do dataset de clientes
df_clientes = df_clientes [['id_cliente', 'nome_completo', 'email', 'cidade', 'estado']]

In [ ]:
# normalizei a coluna email no dataset clientes
df_clientes['email'] = df_clientes['email'].str.replace('#', '@')

df_clientes[~df_clientes['email'].str.contains('@')]

#Verifiquei a nova condição
df_clientes.head()

In [ ]:
# chequei a tabela higienizada
df_clientes.info()
df_clientes.isna().sum()

In [ ]:
# baixei os dados da tabela tratada
df_clientes.to_csv('../dados_tratados/clientes_crm_limpo.csv', index=False, encoding='utf-8-sig')

# Tabela de produtos

In [ ]:
# carreguei os dados do dataset produtos e visualizei as primeiras linhas
df_produtos = pd.read_csv('../dados_brutos/produtos_raw.csv')

df_produtos.head()

In [ ]:
# explorei a estrutura e as estatísticas do dataset de produtos
df_produtos.shape
df_produtos.info()
df_produtos.describe()

In [ ]:
# verifiquei valores nulos no dataset de produtos
df_produtos.isnull().sum()

In [ ]:
# renomeei as colunas para padronização dos dados
df_produtos = df_produtos.rename(columns={
    'name': 'produto',
    'price': 'preco',
    'code': 'id_produto',
    'actual_category': 'categoria_real'

})

# verifiquei condição
df_produtos.head()

In [ ]:
# organizei a ordem das colunas do dataset de produtos
df_produtos = df_produtos[['id_produto', 'produto', 'categoria_real', 'preco']]

# verifiquei condição
df_produtos.head()

In [ ]:
# verifiquei unicidade da chave id_produto
df_produtos['id_produto'].is_unique

In [ ]:
# utilizei da coluna "id_produto" para consultar a quantidade de produtos listados que estavam repetidos
df_produtos['id_produto'].duplicated().sum()

In [ ]:
# para limpar a tabela, removi os prudutos duplicados com o método .drop_duplicates mantendo a primeira linha 
df_produtos = df_produtos.drop_duplicates(subset='id_produto', keep='first')

# verifiquei novamente a consistência da coluna "id_produto"
df_produtos['id_produto'].is_unique

In [ ]:
# padronizei os nomes removendo os espaços e ajustando a capitalização
df_produtos['produto'] = df_produtos['produto'].str.strip()
df_produtos['produto'] = df_produtos['produto'].str.title()

Verificando a consistência da coluna "categoria_real":

In [ ]:
# verifiquei todos os valores da coluna "categoria_real":
df_produtos['categoria_real'].unique()

In [ ]:
# criei um dicionário para padronizar a coluna "categoria_real":
mapa_categorias = {
    # eletronicos
    'eletronicos': 'eletronicos',
    'eletrunicos': 'eletronicos',
    'eletroniscos': 'eletronicos',
    'eletronicoz': 'eletronicos',

    # propulsao
    'propulsao': 'propulsao',
    'propulcao': 'propulsao',
    'prop': 'propulsao',
    'propulssao': 'propulsao',
    'propucao': 'propulsao',
    'propulsam': 'propulsao',
    'propulsão': 'propulsao',

    # ancoragem
    'ancoragem': 'ancoragem',
    'encoragem': 'ancoragem',
    'ancoraguem': 'ancoragem',
    'ancorajm': 'ancoragem',
    'ancorajem': 'ancoragem',
    'ancorajen': 'ancoragem'
}

# reposicionei os valores da coluna "categoria _real" pelo mapa de categorias criado:
df_produtos['categoria_real'] = df_produtos['categoria_real'].replace(mapa_categorias)

In [ ]:
# padronizei a formatação das strings:
df_produtos['categoria_real'] = (
    df_produtos['categoria_real']
    .str.lower()
    .str.strip()
    .str.replace(' ', '')
    .apply(lambda x: unicodedata.normalize('NFKD', x).encode('ascii', 'ignore').decode('utf-8'))
)

In [ ]:
# verificando nova condição:
df_produtos['categoria_real'].unique()


In [ ]:
#Aqui eu formatei a coluna de "preco", uniformizando para o tipo numérico e reorganizando os caracteres
df_produtos['preco'] = (
    df_produtos['preco']
    .str.replace('R$', '', regex=False)
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
    .str.strip()
    .astype(float)
)

In [ ]:
# chequei a tabela higienizada
df_produtos.info()
df_produtos.isna().sum()

In [ ]:
# baixei os dados da tabela tratada
df_produtos.to_csv('../dados_tratados/produtos_crm_limpo.csv', index=False, encoding='utf-8-sig')

# tabela de custos de importação

In [ ]:
# carreguei os dados do dataset de custos e visualizei as primeiras linhas
df_custos = pd.read_json('../dados_brutos/custos_importacao.json')

df_custos.head()

In [ ]:
# explorei a estrutura e as estatísticas do dataset de custos
df_custos.shape
df_custos.info()
df_custos.describe()

In [ ]:
# verifiquei valores nulos no dataset de produtos
df_custos.isnull().sum()

In [ ]:
# renomeei as colunas para padronização dos dados
df_custos = df_custos.rename(columns={
    'product_id': 'id_produto',
    'product_name': 'produto',
    'category': 'categoria',
    'historic_data': 'historico_dados'

})

# verifiquei condição
df_custos.head()

In [ ]:
# verifiquei o tipo de dados na coluna historico_dados
df_custos['historico_dados'].apply(type).value_counts()

In [ ]:
# expandi lista da coluna historico_dados em múltiplas linhas com o .explode
df_custos = df_custos.explode('historico_dados')

In [ ]:
# extrai campos start_date e usd_price da coluna historico_dados
df_custos[['start_date', 'usd_price']] = df_custos['historico_dados'].apply(pd.Series)

In [ ]:
# renomeei as colunas criadas
df_custos = df_custos.rename(columns={
    'start_date': 'data_inicial',
    'usd_price': 'preco_usd'
})

In [ ]:
# eliminei a coluna "historico_dados"
df_custos = df_custos.drop(columns=['historico_dados'])

# verifiquei nova condição
df_custos.head()

In [ ]:
# verifiqeui a consistência da coluna id_produto
df_custos['id_produto'].is_unique

In [ ]:
# removi as células duplicadas 
df_custos = df_custos.drop_duplicates(subset='id_produto', keep='first')

# verifiquei nova condição
df_custos['id_produto'].is_unique

In [ ]:
# verifiquei a coluna categoria
df_custos['categoria'].unique()

In [ ]:
#verifiquei a coluna produto para possível tratamento 
df_custos['produto'].unique()

In [ ]:
# removi os espaços da coluna "produto"
df_custos['produto'] = df_custos['produto'].str.strip()

In [ ]:
# normalizei a coluna "categoria" removendo acentuação
def remover_acento(texto):
    return unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')

df_custos['categoria'] = df_custos['categoria'].apply(remover_acento)

In [ ]:
# transformei o tipo de dado da coluna "data_inicial" para o tipo data
df_custos['data_inicial'] = pd.to_datetime(df_custos['data_inicial'], dayfirst=True)

In [ ]:
# transformei o tipo de dado da coluna "preco_usd" para o tipo float
df_custos['preco_usd'] = df_custos['preco_usd'].astype(float)

In [ ]:
# chequei a tabela higienizada
df_custos.info()
df_custos.isna().sum()

In [ ]:
# baixei os dados da tabela tratada
df_custos.to_csv('../dados_tratados/custos_crm_limpo.csv', index=False, encoding='utf-8-sig')

# tabela de vendas 2023/2024

In [ ]:
# carreguei os dados do dataset de vendas e visualizei as primeiras linhas
df_vendas = pd.read_csv('../dados_brutos/vendas_2023_2024.csv')

df_vendas.head()

In [ ]:
# explorei a estrutura e as estatísticas do dataset de vendas
df_vendas.shape
df_vendas.info()
df_vendas.describe()

In [ ]:
# verifiquei valores nulos no dataset de vendas
df_vendas.isnull().sum()

In [ ]:
# verifiquei possíveis outliers na coluna total
Q1 = df_vendas['total'].quantile(0.25)
Q3 = df_vendas['total'].quantile(0.75)
IQR = Q3 - Q1


limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

outliers = df_vendas[(df_vendas['total'] < limite_inferior) | (df_vendas['total'] > limite_superior)]
print(f"Quantidade de outliers: {len(outliers)}")
outliers.head()

In [ ]:
# renomeei as colunas para padronização dos dados
df_vendas = df_vendas.rename(columns={
    'id_client': 'id_cliente',
    'id_product': 'id_produto',
    'sale_date': 'data_venda'
})

# verifiquei nova condição
df_vendas.head()

In [ ]:
# organizei a ordem das colunas do dataset de vendas
df_vendas = df_vendas[['id', 'id_cliente', 'id_produto', 'data_venda', 'qtd', 'total']]

#Verificando condição
df_vendas.head()

In [ ]:
# verifiquei a consistência das colunas de identificação
df_vendas.duplicated(subset=['id', 'id_produto', 'id_cliente'])

In [ ]:
# formatei o tipo da coluna "data_venda" para data
df_vendas['data_venda'] = pd.to_datetime(
    df_vendas['data_venda'],
    format='mixed',
    dayfirst=True,
    errors='coerce'
)

In [ ]:
# baixei os dados da tabela tratada
df_vendas.info()
df_vendas.isna().sum()

In [ ]:
# baixei os dados da tabela tratada
df_vendas.to_csv('../dados_tratados/vendas_crm_limpo.csv', index=False, encoding='utf-8-sig')

# SQL

In [ ]:
# converti as tabelas para o sqlite
df_clientes = pd.read_csv("../dados_tratados/clientes_crm_limpo.csv")
df_vendas = pd.read_csv("../dados_tratados/vendas_crm_limpo.csv")
df_produtos = pd.read_csv("../dados_tratados/produtos_crm_limpo.csv")
df_custos = pd.read_csv("../dados_tratados/custos_crm_limpo.csv")

In [ ]:
df_clientes.to_sql('clientes_crm_limpo', conn, if_exists='replace', index=False)
df_vendas.to_sql('vendas_crm_limpo', conn, if_exists='replace', index=False)
df_produtos.to_sql('produtos_crm_limpo', conn, if_exists='replace', index=False)
df_custos.to_sql('custos_crm_limpo', conn, if_exists='replace', index=False)

### consultas da tabela clientes:

In [ ]:
# verifiquei a integridade dos dados convertidos na tabela
query = """
SELECT * FROM clientes_crm_limpo
"""

pd.read_sql(query, conn)

In [ ]:
# chequei a quantidade de clientes por estado
query = """
SELECT 
    estado,
    COUNT(*) AS total_clientes
FROM clientes_crm_limpo
GROUP BY estado
ORDER BY total_clientes DESC;
"""
pd.read_sql(query,conn)

In [ ]:
# verifiquei a quantidade de clientes por cidade
query = """
SELECT
    cidade,
    COUNT(*) AS total_clientes
FROM 
    clientes_crm_limpo
GROUP BY 
    cidade
ORDER BY 
    total_clientes DESC;
"""

pd.read_sql(query, conn)

### Consultas da tabela produtos:

In [ ]:
# verifiquei a integridade dos dados convertidos na tabela
query = """
SELECT * FROM produtos_crm_limpo
""" 
pd.read_sql(query, conn)

In [ ]:
# chequei o preço médio por categoria
query = """
SELECT 
    categoria_real,
    ROUND(AVG(preco), 2) AS preco_medio
FROM produtos_crm_limpo
GROUP BY categoria_real
ORDER BY preco_medio DESC;
"""

df_produtos = pd.read_sql(query, conn)

In [ ]:
df_produtos['preco_formatado'] = df_produtos['preco_medio'].apply(
    lambda x: f"R$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
)

df_produtos

In [ ]:
# verifiquei o vlor total das categorias para analisar como essa média ficou distribuida
query = """
SELECT
    categoria_real,
    SUM(preco) AS valor_total
FROM produtos_crm_limpo
GROUP BY categoria_real
ORDER BY valor_total DESC;
"""
df_produtos = pd.read_sql(query, conn)

In [ ]:
df_produtos['preco_formatado'] = df_produtos['valor_total'].apply(
    lambda x: f"R$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
)

df_produtos

In [ ]:
# ordenei os produtos mais caros
query = """
SELECT 
    produto,
    preco
FROM produtos_crm_limpo
ORDER BY preco DESC
LIMIT 10;
"""

df_produtos = pd.read_sql(query, conn)

In [ ]:
df_produtos['preco_formatado'] = df_produtos['preco'].apply(
    lambda x: f"R$ {x:,.2f}".replace(",","x").replace('.', ",").replace("x",".")
)
df_produtos

In [ ]:
# ordenei os produtos mais baratos
query = """
SELECT 
    produto,
    preco
FROM produtos_crm_limpo
ORDER BY preco ASC
LIMIT 10;
"""

df_produtos = pd.read_sql(query, conn)

In [ ]:
df_produtos['preco_formatado'] = df_produtos['preco'].apply(
    lambda x: f"R$ {x:,.2f}".replace(",","x").replace('.', ",").replace("x",".")
)
df_produtos

In [ ]:
# verifiquei o produto mais caro de cata categoria
query = """
SELECT
    categoria_real,
    produto, 
    MAX(preco) AS maior_preco
FROM produtos_crm_limpo
GROUP BY categoria_real;
""" 
df_produtos = pd.read_sql(query, conn)


In [ ]:
df_produtos['preco_formatado'] = df_produtos['maior_preco'].apply(
    lambda x: f"R$ {x:,.2f}".replace(",","x").replace('.', ",").replace("x",".")
)
df_produtos

### Consulta da tabela de custos:

In [ ]:
# verifiquei a integridade dos dados convertidos na tabela
query = """
SELECT * FROM custos_crm_limpo
"""

pd.read_sql(query, conn)

In [ ]:
# verifiquei o preço médio por categoria
query = """
SELECT 
    categoria,
    ROUND(AVG(preco_usd)) AS preco_medio
FROM custos_crm_limpo
GROUP BY categoria
ORDER BY preco_medio DESC
"""
df_custos = pd.read_sql(query, conn)

In [ ]:
df_custos['preco_formatado'] = df_custos['preco_medio'].apply(
    lambda x: f"$ {x:,.2f}"
)

df_custos 

In [ ]:
# verifiquei o vlor total das categorias para analisar como essa média ficou distribuida
query = """
SELECT
    categoria,
    SUM(preco_usd) AS custo_total
FROM custos_crm_limpo
GROUP BY categoria
ORDER BY custo_total DESC;
"""
df_custos = pd.read_sql(query, conn)

In [ ]:
df_custos['custo_formatado'] = df_custos['custo_total'].apply(
    lambda x: f"$ {x:,.2f}"
)

df_custos 

In [ ]:
# verifiqeui os produtos com custo mais alto
query = """
SELECT produto, MAX(preco_usd) as preco_usd
FROM custos_crm_limpo
GROUP BY produto
ORDER BY preco_usd DESC
LIMIT 10
"""
df_custos = pd.read_sql(query, conn)

In [ ]:
df_custos['custo_formatado'] = df_custos['preco_usd'].apply(
    lambda x: f"$ {x:,.2f}"
)

df_custos 

In [ ]:
# verifiquei os produtos com custo mais baixo
query = """
SELECT 
    produto,
    preco_usd
FROM custos_crm_limpo
ORDER BY preco_usd ASC
LIMIT 10;
"""

df_custos = pd.read_sql(query, conn)

In [ ]:
df_custos['custo_formatado'] = df_custos['preco_usd'].apply(
    lambda x: f"$ {x:,.2f}"
)

df_custos 

### Consultas da tabela de vendas 2023/2024

In [ ]:
# verifiquei a integridade dos dados convertidos na tabela
query  = """
SELECT * FROM vendas_crm_limpo
"""
pd.read_sql(query, conn)

In [ ]:
df_produtos.rename(columns={'id':'id_produto'}, inplace=True)

In [ ]:
# uni o dataset de vendas com a tabela de produtos
df_produtos_original = pd.read_csv(
    r"C:\Users\Rangel\OneDrive\Documentos\lh_nautical\dados_tratados\produtos_crm_limpo.csv"
)

In [ ]:
df_vendas_produtos = df_vendas.merge(
    df_produtos_original,
    on='id_produto',
    how='left'
)

df_vendas_produtos.head()

In [ ]:
df_vendas_produtos.to_sql('vendas_produtos', conn, index=False, if_exists='replace')

In [ ]:
# verifiquei os produtos que mais geraramm receita para a empresa
query = """
SELECT 
    produto,
    SUM(total) AS total_vendas
FROM vendas_produtos
GROUP BY produto
ORDER BY total_vendas DESC
LIMIT 10;
"""

melhores_produtos = pd.read_sql(query, conn)

In [ ]:
melhores_produtos['preco_formatado'] = melhores_produtos['total_vendas'].apply(
    lambda x: f"R$ {x:,.2f}".replace(",", "x").replace(".", ",").replace("x", ".")
)

melhores_produtos.head(10)

In [ ]:
# verifiquei a ordem das categorias da maior para a menor receita
query = """
SELECT 
    categoria_real,
    SUM(total) AS total_vendas
FROM vendas_produtos
GROUP BY categoria_real
ORDER BY total_vendas DESC;
"""
top_categoria = pd.read_sql(query, conn)

In [ ]:
top_categoria['preco_formatado'] = top_categoria['total_vendas'].apply(
    lambda x: f"R$ {x:,.2f}".replace(",", "x").replace(".", ",").replace("x", ".")
)
top_categoria.head(10)


In [ ]:
# verifiquei o faturamento da empresa por ano
query ="""
SELECT 
    strftime('%Y', data_venda) AS ano,
    SUM(total) AS total_vendas
FROM vendas_produtos
GROUP BY ano
ORDER BY ano;
"""
vendas_ano = pd.read_sql(query, conn)


In [ ]:
vendas_ano['preco_formatado'] = vendas_ano['total_vendas'].apply(
    lambda x: f"R$ {x:,.2f}".replace(",", "x").replace(".", ",").replace("x", ".")
)
vendas_ano.head(10)


# Questões do desafio:

Questões 1.1 e 1.2:

In [ ]:
df_vendas.to_sql('vendas_2023_2024', conn, index=False, if_exists='replace')

In [ ]:
# Recarregamento da base original
df_vendas = pd.read_csv('../dados_brutos/vendas_2023_2024.csv')

df_vendas.to_sql('vendas_2023_2024', conn, index=False, if_exists='replace')

In [ ]:
query = """
SELECT
    COUNT(*) AS num_linhas,
    6 AS num_colunas,

    MIN(DATE(sale_date)) AS data_min,
    MAX(DATE(sale_date)) AS data_max,

    MIN(total) AS valor_min,
    MAX(total) AS valor_max,
    AVG(total) AS valor_medio

FROM vendas_2023_2024
"""
pd.read_sql(query, conn)

Questão 1.3:

A coluna total apresenta aproximadamente 1.018 valores divergentes em relação à média, representando cerca de 10% do total de 9.895 registros. Esses valores podem indicar a presença de outliers, possivelmente associados a descontos, promoções ou vendas de maior volume, sendo necessário avaliar seu impacto conforme o objetivo da análise
 A tabela Vendas_2023_2024.csv não apresentou valores nulos durante a etapa de verificação, indicando consistência no preenchimento dos dados.
 No entanto, a estrutura da base não se encontra adequada para consultas estruturadas e análises mais robustas. A coluna sale_date, por exemplo, está armazenada como string e apresenta formatos de data inconsistentes, o que exige a padronização para o tipo datetime.
 Dessa forma, torna-se necessária a realização de um processo de higienização e transformação dos dados, visando garantir maior qualidade, consistência e confiabilidade para as análises subsequentes.

Questão 2.1:
Markdown produtos/ Verificando a consistência da coluna "categoria_real"

Questão 2.2: 
7 produtos removidos

Questão 3.1 e 3.2:

In [ ]:
df_custos = pd.read_json('../dados_brutos/custos_importacao.json')

# Desmembrando "historic_data"
df_custos = df_custos.explode('historic_data')

# Renomeando colunas
df_custos = df_custos.rename(columns={
    'product_id': 'id_produto',
    'product_name': 'produto',
    'category': 'categoria',
    'historic_data': 'historico_dados'
})

# Extraindo start_date e usd_price do dicionário
df_custos[['data_inicial', 'preco_usd']] = df_custos['historico_dados'].apply(pd.Series)
df_custos = df_custos.drop(columns=['historico_dados'])

# Normalizando categoria
def remover_acento(texto):
    return unicodedata.normalize('NFKD', str(texto)).encode('ascii', 'ignore').decode('utf-8')

df_custos['categoria'] = df_custos['categoria'].apply(remover_acento)
df_custos['data_inicial'] = pd.to_datetime(df_custos['data_inicial'], dayfirst=True)
df_custos['preco_usd'] = df_custos['preco_usd'].astype(float)

print(f"Total de entradas: {len(df_custos)}") 

df_custos.to_csv('../dados_tratados/custos_crm_limpo.csv', index=False, encoding='utf-8-sig')

Questão 4.1:

In [ ]:
# lendo os arquivos de custos e vendas
vendas  = pd.read_csv('../dados_tratados/vendas_crm_limpo.csv')
custos  = pd.read_csv('../dados_tratados/custos_crm_limpo.csv')

vendas['data_venda'] = pd.to_datetime(vendas['data_venda'])
custos['data_inicial'] = pd.to_datetime(custos['data_inicial'])

# buscando câmbio USD→BRL com API do Banco Central para cada data de venda
def buscar_cambio(data: str) -> float:
    """Retorna a taxa de venda (média) do dólar para uma data no formato YYYY-MM-DD."""
    d = pd.to_datetime(data).strftime('%m-%d-%Y')
    url = (
        f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
        f"CotacaoDolarDia(dataCotacao=@dataCotacao)"
        f"?@dataCotacao='{d}'&$top=1&$format=json&$select=cotacaoVenda"
    )
    try:
        r = requests.get(url, timeout=10)
        dados = r.json().get('value', [])
        if dados:
            return dados[0]['cotacaoVenda']
        return None          # fim de semana / feriado
    except Exception:
        return None

# datas únicas de venda
datas_unicas = vendas['data_venda'].dt.date.unique()
print(f"Buscando câmbio para {len(datas_unicas)} datas...")

cambio_map = {}
for i, d in enumerate(sorted(datas_unicas)):
    taxa = buscar_cambio(str(d))
    cambio_map[str(d)] = taxa
    if i % 50 == 0:
        print(f"  {i}/{len(datas_unicas)} datas processadas...")
    time.sleep(0.15)   

# preenchendo fins de semana/feriados com o último câmbio disponível 
datas_sorted = sorted(cambio_map.keys())
ultima_taxa = None
for d in datas_sorted:
    if cambio_map[d] is not None:
        ultima_taxa = cambio_map[d]
    else:
        cambio_map[d] = ultima_taxa  

print("Câmbio carregado com sucesso!")

# juntando câmbio nas vendas
vendas['data_str']   = vendas['data_venda'].dt.strftime('%Y-%m-%d')
vendas['taxa_cambio'] = vendas['data_str'].map(cambio_map)

# juntando o custo unitário (USD) de cada produto
custos_sorted = custos.sort_values('data_inicial')

def custo_na_data(id_prod, data_venda):
    hist = custos_sorted[
        (custos_sorted['id_produto'] == id_prod) &
        (custos_sorted['data_inicial'] <= data_venda)
    ]
    if hist.empty:
        return None
    return hist.iloc[-1]['preco_usd']

print("Calculando custo unitário por transação (pode demorar)...")
vendas['custo_usd_unit'] = vendas.apply(
    lambda r: custo_na_data(r['id_produto'], r['data_venda']), axis=1
)

# calculando custo total BRL e prejuízo
vendas['custo_brl_total'] = vendas['custo_usd_unit'] * vendas['taxa_cambio'] * vendas['qtd']
vendas['prejuizo']        = (vendas['custo_brl_total'] - vendas['total']).clip(lower=0)

# salvando no SQLite para a query SQL 
conn = sqlite3.connect(":memory:")
vendas.to_sql('vendas_cambio', conn, if_exists='replace', index=False)

# agregando por id_produto 
query = """
SELECT
    id_produto,
    ROUND(SUM(total), 2)                                      AS receita_total,
    ROUND(SUM(prejuizo), 2)                                   AS prejuizo_total,
    ROUND(SUM(prejuizo) * 100.0 / NULLIF(SUM(total), 0), 4)  AS pct_perda
FROM vendas_cambio
WHERE custo_usd_unit IS NOT NULL
  AND taxa_cambio   IS NOT NULL
GROUP BY id_produto
HAVING SUM(prejuizo) > 0
ORDER BY pct_perda DESC;
"""
resultado = pd.read_sql(query, conn)
print(resultado.head(10))
print(f"\nProduto com MAIOR % de perda: id_produto = {resultado.iloc[0]['id_produto']}")
print(f"  % de perda: {resultado.iloc[0]['pct_perda']:.4f}%")

Questão 4.2: O produto com MAIOR % de perda: id_produto = 72.0

Questão 4.3: A cotação USD→BRL foi obtida da API PTAX do Banco Central, usando a venda do dia da transação. Para fins de semana e feriados, utilizei o último câmbio disponível.
Prejuízo: calculei o custo em reais de cada venda (preço em dólar × câmbio × quantidade). Se esse custo for maior que o valor da venda, consideramos a diferença como prejuízo. OBS: Transações sem custo disponível foram excluídas.

Questão 5:

In [ ]:
vendas = pd.read_csv('../dados_tratados/vendas_crm_limpo.CSV')
produtos = pd.read_csv('../dados_tratados/produtos_crm_limpo.CSV')

In [ ]:
vendas_produtos = vendas.merge(produtos[['id_produto','categoria_real']], on='id_produto', how='left')


In [ ]:
df_vendas   = pd.read_csv('../dados_tratados/vendas_crm_limpo.csv')
df_produtos = pd.read_csv('../dados_tratados/produtos_crm_limpo.csv')

In [ ]:
df_vendas.to_sql('vendas_crm_limpo', conn, if_exists='replace', index=False)
df_produtos.to_sql('produtos_crm_limpo', conn, if_exists='replace', index=False)

In [ ]:
tabelas = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print("Tabelas disponíveis:")
print(tabelas)


In [ ]:
query = """
WITH categorias_limpas AS (
    SELECT
        id_produto,
        CASE
            WHEN LOWER(REPLACE(categoria_real, ' ', '')) IN (
                'eletronicos','eletrunicos','eletroniscos','eletronicoz'
            ) THEN 'eletronicos'
            WHEN LOWER(REPLACE(categoria_real, ' ', '')) IN (
                'propulsao','propulcao','prop','propulssao','propucao','propulsam'
            ) THEN 'propulsao'
            ELSE 'ancoragem'
        END AS categoria
    FROM produtos_crm_limpo
),
metricas_clientes AS (
    SELECT
        v.id_cliente,
        ROUND(SUM(v.total), 2)                         AS faturamento_total,
        COUNT(DISTINCT v.id)                           AS frequencia,
        ROUND(SUM(v.total) / COUNT(DISTINCT v.id), 2) AS ticket_medio,
        COUNT(DISTINCT c.categoria)                    AS diversidade_categorias
    FROM vendas_crm_limpo v
    JOIN categorias_limpas c ON v.id_produto = c.id_produto
    GROUP BY v.id_cliente
),
top10 AS (
    SELECT
        id_cliente,
        ticket_medio,
        diversidade_categorias
    FROM metricas_clientes
    WHERE diversidade_categorias >= 3
    ORDER BY ticket_medio DESC, id_cliente ASC
    LIMIT 10
)
SELECT
    c.categoria,
    SUM(v.qtd) AS total_itens
FROM vendas_crm_limpo v
JOIN categorias_limpas c ON v.id_produto = c.id_produto
WHERE v.id_cliente IN (SELECT id_cliente FROM top10)
GROUP BY c.categoria
ORDER BY total_itens DESC
LIMIT 1;
"""

resultado_q5 = pd.read_sql(query, conn)
print(resultado_q5)

Questão 6.1:

In [ ]:
query = """
WITH RECURSIVE calendario AS (
    SELECT DATE(MIN(data_venda)) AS dia
    FROM vendas_crm_limpo

    UNION ALL

    SELECT DATE(dia, '+1 day')
    FROM calendario
    WHERE dia < (SELECT DATE(MAX(data_venda)) FROM vendas_crm_limpo)
),

calendario_ptbr AS (
    SELECT
        dia,
        CASE CAST(strftime('%w', dia) AS INTEGER)
            WHEN 0 THEN 'Domingo'
            WHEN 1 THEN 'Segunda-feira'
            WHEN 2 THEN 'Terca-feira'
            WHEN 3 THEN 'Quarta-feira'
            WHEN 4 THEN 'Quinta-feira'
            WHEN 5 THEN 'Sexta-feira'
            WHEN 6 THEN 'Sabado'
        END AS dia_semana,
        CAST(strftime('%w', dia) AS INTEGER) AS num_dia
    FROM calendario
),

vendas_diarias AS (
    SELECT
        c.dia,
        c.dia_semana,
        c.num_dia,
        COALESCE(SUM(v.total), 0) AS valor_venda
    FROM calendario_ptbr c
    LEFT JOIN vendas_crm_limpo v ON DATE(v.data_venda) = c.dia
    GROUP BY c.dia, c.dia_semana, c.num_dia
)

SELECT
    dia_semana,
    COUNT(dia)                 AS total_dias,
    ROUND(AVG(valor_venda), 2) AS media_vendas
FROM vendas_diarias
GROUP BY dia_semana, num_dia
ORDER BY media_vendas ASC;
"""

resultado_q6 = pd.read_sql(query, conn)
print(resultado_q6)

Questão 6.2:
0        Domingo         105    3229614.16

Questão 6.3:
Usar uma tabela de datas é importante porque a tabela de vendas só registra os dias em que houve alguma venda. Se você fizer a análise direto nela, acaba ignorando os dias em que a loja não vendeu nada, o que distorce os resultados. Isso faz com que a média fique artificialmente mais alta, já que considera apenas os dias com venda.

Questão 7.1:

In [ ]:
PRODUTO = 'Motor De Popa Yamaha Evo Dash 155Hp'   

vendas = pd.read_csv('../dados_tratados/vendas_crm_limpo.csv',
                     parse_dates=['data_venda'])
produtos = pd.read_csv('../dados_tratados/produtos_crm_limpo.csv')

# juntando para obter nome do produto
vp = vendas.merge(produtos[['id_produto', 'produto']], on='id_produto', how='left')
vp_filtrado = vp[vp['produto'].str.strip().str.title() == PRODUTO].copy()

# série diária
periodo_completo = pd.date_range(
    start=vp_filtrado['data_venda'].min(),
    end='2024-01-31',
    freq='D'
)
serie_diaria = (
    vp_filtrado.groupby('data_venda')['qtd'].sum()
    .reindex(periodo_completo, fill_value=0)
    .rename_axis('data')
    .reset_index(name='qtd_real')
)

# treino até 31/12/2023 | Teste = Janeiro 2024
treino = serie_diaria[serie_diaria['data'] <= '2023-12-31'].copy()
teste  = serie_diaria[serie_diaria['data'] >= '2024-01-01'].copy()

# média Móvel dos últimos 7 dias
historico_completo = serie_diaria.set_index('data')['qtd_real']

previsoes = []
for data in teste['data']:
    # usando apenas os 7 dias ANTERIORES à data prevista 
    janela = historico_completo[
        (historico_completo.index < data) &
        (historico_completo.index >= data - pd.Timedelta(days=7))
    ]
    previsao = janela.mean() if len(janela) > 0 else 0
    previsoes.append(round(previsao, 4))

teste = teste.copy()
teste['previsao'] = previsoes

# MAE 
mae = np.mean(np.abs(teste['qtd_real'] - teste['previsao']))

primeira_semana = teste[teste['data'] <= '2024-01-07']
soma_semana1 = round(primeira_semana['previsao'].sum())

print("=" * 55)
print(f"Produto analisado : {PRODUTO}")
print(f"MAE do modelo     : {mae:.4f} unidades/dia")
print(f"Soma prevista (01/01 a 07/01/2024): {soma_semana1} unidades")
print("=" * 55)
print("\nPrevisão diária - Janeiro 2024:")
print(teste[['data','qtd_real','previsao']].to_string(index=False))

Questão 7.2: Soma prevista (01/01 a 07/01/2024): 3 unidades

Questão 7.3: O baseline foi construído de forma simples: para cada dia de janeiro de 2024, a previsão é a média das vendas dos 7 dias anteriores, sempre usando apenas dados já observados e sem incluir o próprio dia ou qualquer informação futura. Para evitar vazamento de dados, garantimos que o cálculo considere apenas valores anteriores à data prevista, e o treino usa dados até 31/12/2023, com as previsões de janeiro sendo feitas dia a dia, sempre olhando para trás. Como limitação, esse modelo é bem básico e não capta padrões mais complexos, como sazonalidade ou tendências ao longo do tempo, então pode errar em períodos com comportamento diferente

Questão 8:

In [ ]:
# matriz Usuário × Produto
matriz = (
    vendas.groupby(['id_cliente', 'id_produto'])['qtd']
    .sum()
    .unstack(fill_value=0)
    .clip(upper=1)
)

print(f"Matriz: {matriz.shape[0]} clientes × {matriz.shape[1]} produtos")

# similaridade de Cosseno com numpy
M = matriz.values.T.astype(float) 

# norma de cada produto
normas = np.linalg.norm(M, axis=1, keepdims=True)
normas[normas == 0] = 1  # evita divisão por zero

# normalizando matriz
M_norm = M / normas

# similaridade = produto interno dos vetores normalizados
sim_matrix = np.dot(M_norm, M_norm.T)

sim_df = pd.DataFrame(
    sim_matrix,
    index=matriz.columns,
    columns=matriz.columns
)

# identificando o id do produto de referência
PRODUTO_REF = 'Gps Garmin Vortex Maré Drift'

produtos['produto_norm'] = produtos['produto'].str.strip().str.title()

match = produtos[produtos['produto_norm'] == PRODUTO_REF]
if match.empty:
    match = produtos[produtos['produto_norm'].str.contains('Garmin Vortex', na=False)]

id_ref = match.iloc[0]['id_produto']
print(f"\nProduto de referência: '{match.iloc[0]['produto']}' (id_produto={id_ref})")

# top 5 mais similares 
similares = (
    sim_df[id_ref]
    .drop(index=id_ref)
    .sort_values(ascending=False)
    .head(5)
    .reset_index()
)
similares.columns = ['id_produto', 'similaridade']
similares = similares.merge(produtos[['id_produto', 'produto']], on='id_produto', how='left')

print("\n=== Top 5 produtos mais similares ===")
print(similares[['id_produto', 'produto', 'similaridade']].to_string(index=False))
print(f"\nResposta Q8.2 — id_produto com MAIOR similaridade: {similares.iloc[0]['id_produto']}")
print(f"  Produto : {similares.iloc[0]['produto']}")
print(f"  Score   : {similares.iloc[0]['similaridade']:.6f}")